## 3D Fiber Segmentation

### Objective:
Segment fibers in a 3D image from MicroCT scans using different methods of continuity such as the one used in 2D and based on Persistence Length.

In [ ]:
# Librarys and functions

import os
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from PIL import Image
from skimage.measure import label
from scipy.spatial import cKDTree
from skimage.morphology import remove_small_objects
from skimage.color import label2rgb, gray2rgb
from skimage.measure import label, regionprops
from skimage.morphology import skeletonize
from scipy.ndimage import binary_closing, binary_opening
from skimage.util import invert, img_as_float
from skan import Skeleton, summarize
from itertools import combinations
from scipy.ndimage import distance_transform_edt
from skimage.segmentation import watershed
from skimage.io import imread
from skimage import filters, restoration
from tqdm import tqdm
from IPython.display import clear_output
from skimage.filters import threshold_otsu

In [ ]:
# Implemented functions

def binary_to_rgb(binary_img):
    mask = binary_img.astype(bool)
    rgb_img = np.zeros((*mask.shape, 3), dtype=np.uint8)
    rgb_img[mask] = (255, 0, 0)
    return rgb_img

def binary_to_gray(binary_img, fg=255, bg=0):
    return np.where(binary_img, fg, bg).astype(np.uint8)

def junction_mask_3d(skeleton_3d):
    sk = Skeleton(skeleton_3d)
    labels = np.zeros(skeleton_3d.shape, dtype=np.uint8)

    # Label all skeleton voxels as 1 (non-junction by default)
    labels[skeleton_3d] = 1

    # Label junction voxels as 2
    j_ids = np.where(sk.degrees >= 3)[0]
    if j_ids.size:
        coords = np.round(sk.coordinates[j_ids]).astype(int)  # (z,y,x)
        z = np.clip(coords[:, 0], 0, skeleton_3d.shape[0] - 1)
        y = np.clip(coords[:, 1], 0, skeleton_3d.shape[1] - 1)
        x = np.clip(coords[:, 2], 0, skeleton_3d.shape[2] - 1)
        labels[z, y, x] = 2

    return labels

def filter_skeleton(skeleton_3d_img, tam_min):
    print("------------FILTERING THE SKELETON----------")
    print("     - Removing branches with ", tam_min, " or less voxels.")
    # Copying the original skeleton and making the object
    _skeleton_3d = skeleton_3d_img.copy()
    skeleton_obj = Skeleton(_skeleton_3d)

    # Getting the skeleton information
    summary = summarize(skeleton_obj, separator='-')

    # Getting the junction points
    junction_points_ids = np.where(skeleton_obj.degrees >= 3)[0]

    print("     -", len(junction_points_ids), " junction points found.")

    n_rm = 0

    # Iterate througth the junction points
    for junction_id in junction_points_ids:

        # Get the branches connected to the jp
        connected_branches = summary[(summary['node-id-src'] == junction_id) | (summary['node-id-dst'] == junction_id)]

        # Erase the branches with less than 5 voxels and that are not connected on two junction points
        for _, branch in connected_branches.iterrows():
            branch_id = int(branch.name)
            branch_voxels = skeleton_obj.path_coordinates(branch_id)
            branch_len = len(branch_voxels)

            # Classify the extreme nodes of the branch
            src = int(branch['node-id-src']); dst = int(branch['node-id-dst'])
            deg_src = int(skeleton_obj.degrees[src]); deg_dst = int(skeleton_obj.degrees[dst])
            is_jj = (deg_src >= 3 and deg_dst >= 3)
            is_je = ((deg_src >= 3 and deg_dst == 1) or (deg_dst >= 3 and deg_src == 1))

            # Remover apenas spurs J-E curtos
            if branch_len <= tam_min and is_je and not is_jj:
                n_rm += 1
                z, y, x = branch_voxels[1:].T  # preserva o voxel do nó
                _skeleton_3d[z, y, x] = False
        
    print("     -", n_rm, " branches removed.")
    print("------------FILTERING ENDED---------------\n")
    
    return _skeleton_3d

def dir_vec(junc, pt):
    junc = np.asarray(junc, float)
    pt   = np.asarray(pt,   float)
    v = pt - junc
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def best_straight_pair(vectors):
    V = np.vstack(vectors)             # (n,3) já normalizados
    cos = V @ V.T                      # cossenos
    np.fill_diagonal(cos, +np.inf)     # ignora diagonal
    i, j = np.unravel_index(np.argmin(cos), cos.shape)  # menor cos ~ -1 => ~180°
    ang = np.degrees(np.arccos(np.clip(cos[i, j], -1, 1)))
    return (i, j, ang)

def prune_skeleton(skeleton_3d_img, local_search, local_range):
    # Copying the original skeleton and making the object
    _skeleton_3d = skeleton_3d_img.copy()
    skeleton_obj = Skeleton(_skeleton_3d)

    # Getting the skeleton information
    summary = summarize(skeleton_obj, separator='-')

    # Getting the junction points
    junction_points_ids = np.where(skeleton_obj.degrees >= 3)[0]

    print(len(junction_points_ids), " junction points found.")

    # Iterate througth the junction points
    for junction_id in junction_points_ids:

        # Get the coordinates of the jp
        junction_coords = skeleton_obj.coordinates[junction_id]

        print("-> Analysing junction point {", junction_coords, "}")

        # Get the branches connected to the jp
        connected_branches = summary[(summary['node-id-src'] == junction_id) | (summary['node-id-dst'] == junction_id)]

        # Garantee minimum of three branches
        if len(connected_branches) < 3:
            continue

        vectors = []
        branches_ids = []
        for _, branch in connected_branches.iterrows():
            
            # Enter if local_search is 1, i.e., the decision is made based on the local continuity, not the global
            if local_search:
                branch_voxels = skeleton_obj.path_coordinates(int(branch.name))
                len_branch = len(branch_voxels)

                # Localize the start/end of the branch that end/start at the junction point

                if branch['node-id-src'] == junction_id:                                # Is the junction point the start of the branch?
                    if len_branch > local_range:                                        # Forces local analysis
                        other_coords = branch_voxels[local_range]                       # Get the coordinates of the voxel in the 
                                                                                        #   local distance as the endpoint
                    else:
                        other_node_id = branch['node-id-dst']                           # Get the endpoint
                        other_coords = skeleton_obj.coordinates[int(other_node_id)]     # Get the coordinates of the endpoint
                else:                                                                   # The junction point is the end of the branch
                    if len_branch > local_range:                                        # Forces local analysis
                        other_coords = branch_voxels[-(local_range-1)]                  # Get the coordinates of the voxel in the 
                                                                                        #   local distance as the startpoint
                    else:
                        other_node_id = branch['node-id-src']                           # Get the startpoint
                        other_coords = skeleton_obj.coordinates[int(other_node_id)]     # Get the coordinates of the startpoint
            else:
                other_node_id = branch['node-id-dst'] if branch['node-id-src'] == junction_id else branch['node-id-src']
                junc_is_src = 1 if branch['node-id-src'] == junction_id else 0
                other_coords = skeleton_obj.coordinates[int(other_node_id)]

            vectors.append(dir_vec(junction_coords, other_coords))
            branches_ids.append(int(branch.name))
        
        i, j, ang = best_straight_pair(vectors)

        print("     Branches checked: ", branches_ids)
        print("     Chosen branches: {", branches_ids[i], branches_ids[j], "} w/ angle = ", round(ang, 2))

        if ang > 90:
            branch_to_prune = int(np.delete(branches_ids, [i,j])[0])

            branch_to_prune_voxels = skeleton_obj.path_coordinates(branch_to_prune)

            if np.array_equal(branch_to_prune_voxels[0], junction_coords):
                coords_to_set_zero = tuple(branch_to_prune_voxels[1])
            else:
                coords_to_set_zero = tuple(branch_to_prune_voxels[-2])
            
            print("     Prunning Branch {", branch_to_prune, "} at coord {", coords_to_set_zero, "} \n\n")

            _skeleton_3d[coords_to_set_zero] = 0

        else:

            print("     Best angle smaller or equal to 90°. Prunning all the branches connected to the junction point.")
            for branch_to_prune in branches_ids:
                branch_to_prune_voxels = skeleton_obj.path_coordinates(branch_to_prune)

                if np.array_equal(branch_to_prune_voxels[0], junction_coords):
                    coords_to_set_zero = tuple(branch_to_prune_voxels[1])
                else:
                    coords_to_set_zero = tuple(branch_to_prune_voxels[-2])
            
                print("     Prunning Branch {", branch_to_prune, "} at coord {", coords_to_set_zero, "}")

                _skeleton_3d[coords_to_set_zero] = 0
            print("\n\n")
    
    return _skeleton_3d

def label_fibers_watershed(binary, markers):
    """
    Aplica distância euclidiana 3D e Watershed para propagar rótulos do esqueleto.
    Parâmetros:
      - binary: volume binário (bool ou 0/1)
      - markers: volume de marcadores inteiros (mesmo shape de binary)
    Retorna:
      - fibras_rotuladas: rótulos propagados pelo Watershed
      - distancia: distância euclidiana 3D do binário
    """
    binary = binary.astype(bool)
    if binary.shape != markers.shape:
        raise ValueError("binary e markers devem ter o mesmo shape.")

    distancia = distance_transform_edt(binary)
    fibras_rotuladas = watershed(-distancia, markers=markers, mask=binary)
    return fibras_rotuladas



In [ ]:
# Importing imagem
# Image options:
#   "./fiber_img/data/PPGF40.tif"
#   "./fiber_img/data/PPGF40HNT0-1.0.tif"
#   "./fiber_img/data/PPGF40HNT0-1.5.tif"
#   "./fiber_img/data/PPGF40HNT0-5.tif"

filename = "./fiber_img/data/PPGF40HNT0-1.0.tif"
raw_tiff_img = tifffile.imread(filename)

# Converting to 8bits gray scale
raw_tiff_img = (raw_tiff_img / np.max(raw_tiff_img) * 255).astype(np.uint8)

In [ ]:
# Details of the image
print("Selected image: ", filename)
print("Dimensions raw image:",
      " \n X: ", raw_tiff_img.shape[0], 
      " \n Y: ", raw_tiff_img.shape[1], 
      " \n Z: ", raw_tiff_img.shape[2])

# Selecting only the slices with no shadows
if filename == "./fiber_img/data/PPGF40HNT0-1.0.tif":
    selected_tiff_img = raw_tiff_img[:, 125:940, 130:535]

print("Dimensions selected image (no shadows): ",
      " \n X: ", selected_tiff_img.shape[0], 
      " \n Y: ", selected_tiff_img.shape[1], 
      " \n Z: ", selected_tiff_img.shape[2])

# Axis plot
plt.figure(dpi=200)
plt.suptitle("Slices examples through")
plt.subplot(1,3,1)
plt.imshow(selected_tiff_img[500,:,:], cmap='gray')
plt.title("YZ axis")

plt.subplot(1,3,2)
plt.imshow(selected_tiff_img[:,400,:], cmap='gray')
plt.title("XZ axis")

plt.subplot(1,3,3)
plt.imshow(selected_tiff_img[:,:,200], cmap='gray')
plt.title("XY axis")

plt.tight_layout()
plt.show()

In [ ]:
# Image binarization

# Choosing threshold with Otsu criterium
binary_threshold = threshold_otsu(selected_tiff_img)

# Verifying the chosen threshold looking at the voxels intensity distributions
histogram_intensity, bins = np.histogram(selected_tiff_img.ravel(), bins=256, range=(0, 256))

plt.figure(figsize=(10, 6))
plt.title("Histograma do Volume 3D")
plt.xlabel("Nível de Cinza (Intensidade do Voxel)")
plt.ylabel("Frequência")
plt.plot(histogram_intensity, color='gray')
plt.grid(True, linestyle='--', alpha=0.6)
plt.axvline(x=binary_threshold, color='r', linestyle='--')
plt.show()

binary_img = selected_tiff_img > binary_threshold

In [ ]:
# Binarization results

# Axis plot
plt.figure(dpi=200)
plt.suptitle("Binarization results")
plt.subplot(1,3,1)
plt.imshow(binary_img[10,:,:], cmap='gray')
plt.title("ZY axis")

plt.subplot(1,3,2)
plt.imshow(binary_img[:,10,:], cmap='gray')
plt.title("ZX axis")

plt.subplot(1,3,3)
plt.imshow(binary_img[:,:,10], cmap='gray')
plt.title("YX axis")

plt.tight_layout()
plt.show()

# Visualizing w/ Napari
# tifffile.imwrite("./fiber_img/3d-res/binary_img.tif", binary_img.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/binary_img.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/binary_img.tif

In [ ]:
# Selecting a 3D cubic section of the block for tests
n_pixels = 200

mean_X = round(selected_tiff_img.shape[0]/2)
mean_Y = round(selected_tiff_img.shape[1]/2)
mean_Z = round(selected_tiff_img.shape[2]/2)

n_pixels_each_side = round(n_pixels/2)

binary_cropped = binary_img[mean_X-n_pixels_each_side:mean_X+n_pixels_each_side,
                                mean_Y-n_pixels_each_side:mean_Y+n_pixels_each_side,
                                mean_Z-n_pixels_each_side:mean_Z+n_pixels_each_side]

# Visualizing w/ Napari
# tifffile.imwrite("./fiber_img/3d-res/binary_cropped.tif", binary_cropped.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/binary_cropped.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/binary_cropped.tif

In [ ]:
# Skeletonizing full image (not recommend for tests)

# skeleton_img = skeletonize(binary_img)

# Visualizing w/ Napari
# tifffile.imwrite("./fiber_img/3d-res/skeleton_img.tif", skeleton_img.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/skeleton_img.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/skeleton_img.tif

In [ ]:
# Skeletonizing cropped image for tests

skeleton_cropped = skeletonize(binary_cropped, method='lee')

# Saving figure on the server, copying to local machine and Visualizing w/ Napari
tifffile.imwrite("./fiber_img/3d-res/skeleton_cropped.tif", skeleton_cropped.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/skeleton_cropped.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/skeleton_cropped.tif

In [ ]:
# Labeling the skeleton (not segmented) to see the initial composition

labeled_cropped = label(skeleton_cropped)

# Saving figure on the server, copying to local machine and Visualizing w/ Napari
tifffile.imwrite("./fiber_img/3d-res/labeled_cropped.tif", labeled_cropped.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/labeled_cropped.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/labeled_cropped.tif --layer-type labels

In [ ]:
# Filtering the skeleton and marking the junction points

filtered_skeleton = filter_skeleton(skeleton_cropped, 5)

# Saving figure on the server, copying to local machine and Visualizing w/ Napari
tifffile.imwrite("./fiber_img/3d-res/filtered_skeleton.tif", filtered_skeleton.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/filtered_skeleton.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/filtered_skeleton.tif

marked_skeleton = junction_mask_3d(filtered_skeleton)

# Saving figure on the server, copying to local machine and Visualizing w/ Napari
tifffile.imwrite("./fiber_img/3d-res/marked_skeleton.tif", marked_skeleton.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/marked_skeleton.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/marked_skeleton.tif --layer-type labels

In [ ]:
# Prunning the skeleton by naive continuity
prunned_skeleton = prune_skeleton(filtered_skeleton, local_search=False, local_range=10)

In [ ]:
labeled_skeleton = label(prunned_skeleton)

# Saving figure on the server, copying to local machine and Visualizing w/ Napari
tifffile.imwrite("./fiber_img/3d-res/labeled_skeleton.tif", labeled_skeleton.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/labeled_skeleton.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/labeled_skeleton.tif --layer-type labels

In [ ]:
labeled_fibers = label_fibers_watershed(binary_cropped, labeled_skeleton)

# Saving figure on the server, copying to local machine and Visualizing w/ Napari
tifffile.imwrite("./fiber_img/3d-res/labeled_fibers.tif", labeled_fibers.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/labeled_fibers.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/labeled_fibers.tif --layer-type labels

## Cálculo das métricas

### 1. Cálculo do comprimento das fibras
### 2. Cálculo da orientação das fibras

In [ ]:
def calculate_fiber_length(labeled_skeleton):
    regs = regionprops(labeled_skeleton)
    results =[]
    for r in regs:
        n_voxels = r.num_pixels
        x0,y0,z0,x1,y1,z1 = r.bbox
        length = np.sqrt((x0-x1)**2+(y0-y1)**2+(z0-z1)**2)
        results.append({"label": r.label,
                        "n_voxels": n_voxels,
                        "length": length})
    return pd.DataFrame(results)

def calculate_fiber_orientation(labeled_skeleton):
    regs = regionprops(labeled_skeleton)
    results =[]
    for r in regs:
        if r.area < 2:
            continue
        coords_zyx = r.coords.astype(float)
        if coords_zyx.shape[0] < 2:
            continue
        coords_zyx -= coords_zyx.mean(axis=0, keepdims=True)
        
        try:
            _, S, Vt = np.linalg.svd(coords_zyx, full_matrices=False)
        except np.linalg.LinAlgError:
            continue
        if S[0] == 0:
            continue

        v_zyx = Vt[0]
        v_zyx /= np.linalg.norm(v_zyx)

        if v_zyx[0] < 0:
            v_zyx = -v_zyx

        vx, vy, vz = v_zyx[0], v_zyx[1], v_zyx[2]

        ang_x = np.degrees(np.arccos(np.clip(vx, -1, 1)))
        ang_y = np.degrees(np.arccos(np.clip(vy, -1, 1)))
        ang_z = np.degrees(np.arccos(np.clip(vz, -1, 1)))

        results.append({"label": r.label,
                        "ang_to_x": ang_x,
                        "ang_to_y": ang_y,
                        "ang_to_z": ang_z})
    return pd.DataFrame(results)

In [ ]:
# Máscara dos ramos (componentes) do labeled_skeleton com 1 ou 2 voxels
regs = regionprops(labeled_skeleton)
small_labels = [r.label for r in regs if r.area in (1, 2, 3, 4, 5, 6)]

big_labels = [r.label for r in regs if r.area not in (1, 2, 3, 4, 5, 6)]

small_branches_mask = np.isin(labeled_skeleton, big_labels)  # bool
small_branches_mask = label(small_branches_mask)

print(f"{len(regs)} ramos encontrados")
print(f"{len(small_labels)} ramos com 1 ou 2 voxels.")
print(f"{len(big_labels)} ramos com mais de 1 ou 2 voxels.")

# Exemplo de salvamento (descomente se quiser)
tifffile.imwrite("./fiber_img/3d-res/small_branches_mask.tif", small_branches_mask.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/small_branches_mask.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/small_branches_mask.tif --layer-type labels

labeled_fibers_mask_test = label_fibers_watershed(binary_cropped, small_branches_mask)

tifffile.imwrite("./fiber_img/3d-res/labeled_fibers_mask_test.tif", labeled_fibers_mask_test.astype(np.uint8) * 255)
# scp homero_castro@litc59:/home/homero_castro/toronto/fiber_img/3d-res/labeled_fibers_mask_test.tif C:\Users\Homero\documents\LITC\Toronto_Fibers\fiber_img\3d-res
# napari ./fiber_img/3d-res/labeled_fibers_mask_test.tif --layer-type labels


In [ ]:
import heapq

# Funções para medir comprimento e orientação das fibras a partir de labeled_fibers

def _neighbor_offsets_with_weights(voxel_size=(1.0, 1.0, 1.0)):
    # Gera offsets 26-vizinhos e pesos euclidianos ponderados pelos voxels
    sz, sy, sx = voxel_size
    offsets, weights = [], []
    for dz in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                if dz == 0 and dy == 0 and dx == 0:
                    continue
                offsets.append((dz, dy, dx))
                weights.append(np.sqrt((dz * sz) ** 2 + (dy * sy) ** 2 + (dx * sx) ** 2))
    return np.asarray(offsets, dtype=int), np.asarray(weights, dtype=float)


def _skeleton_longest_geodesic_length(skel_bool, voxel_size=(1.0, 1.0, 1.0)):
    # Calcula o maior caminho geodésico no esqueleto (grafo 26-conexo ponderado)
    coords = np.argwhere(skel_bool)
    n = coords.shape[0]
    if n <= 1:
        return 0.0

    offsets, w = _neighbor_offsets_with_weights(voxel_size)
    coord_to_idx = {tuple(c): i for i, c in enumerate(coords)}
    adj = [[] for _ in range(n)]
    for i, c in enumerate(coords):
        for k, off in enumerate(offsets):
            nb = tuple(c + off)
            j = coord_to_idx.get(nb, None)
            if j is not None:
                adj[i].append((j, float(w[k])))

    # Dijkstra simples

    def dijkstra(start):
        dist = np.full(n, np.inf, dtype=float)
        dist[start] = 0.0
        h = [(0.0, start)]
        while h:
            d, u = heapq.heappop(h)
            if d > dist[u]:
                continue
            for v, ww in adj[u]:
                nd = d + ww
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(h, (nd, v))
        return dist

    degrees = np.array([len(nei) for nei in adj], dtype=int)
    endpoints = np.flatnonzero(degrees == 1)

    # Escolhe um nó inicial
    start = int(endpoints[0]) if endpoints.size else 0
    d1 = dijkstra(start)
    u = int(np.argmax(d1))
    d2 = dijkstra(u)
    v = int(np.argmax(d2))
    return float(d2[v])

def calcular_tamanho_fibras(labeled_fibers, voxel_size=(1.0, 1.0, 1.0), min_voxels=2):
    """
    Retorna um DataFrame com o comprimento (maior caminho geodésico do esqueleto) de cada fibra.
    voxel_size = (sz, sy, sx) permite medir em unidades físicas.
    """
    regs = regionprops(labeled_fibers)
    resultados = []
    for r in regs:
        if r.area < min_voxels:
            continue
        z0, y0, x0, z1, y1, x1 = r.bbox
        sub = (labeled_fibers[z0:z1, y0:y1, x0:x1] == r.label)
        # Esqueletização 3D
        skel = skeletonize(sub, method='lee')
        length = _skeleton_longest_geodesic_length(skel, voxel_size=voxel_size)
        resultados.append({"label": int(r.label), "comprimento": float(length), "voxel_count": int(r.area)})
    return pd.DataFrame(resultados)

# def calcular_tamanho_fibras(labeled_skeleton, voxel_size=(1.0, 1.0, 1.0), min_voxels=2):
#     regs = regionprops(labeled_skeleton)
#     for r in regs:
#         if r.num_pixels < min_voxels:



def calcular_orientacao_fibras(labeled_fibers, min_voxels=2):
    """
    Retorna um DataFrame com orientação principal (vetor unitário) e ângulos:
      - theta_deg: ângulo polar em relação ao eixo Z (0..180)
      - phi_deg: azimute no plano XY (-180..180)
    """
    regs = regionprops(labeled_fibers)
    linhas = []
    for r in regs:
        if r.area < min_voxels:
            continue
        z0, y0, x0, z1, y1, x1 = r.bbox
        sub = (labeled_fibers[z0:z1, y0:y1, x0:x1] == r.label)
        skel = skeletonize(sub, method='lee')
        sk_coords = np.argwhere(skel)
        if sk_coords.shape[0] < 2:
            # Fallback: usa voxels da região
            sk_coords = r.coords - np.array([z0, y0, x0])

        if sk_coords.shape[0] < 10:
            continue

        # Converte para unidades físicas
        pts = sk_coords.astype(float)
        pts -= pts.mean(axis=0, keepdims=True)

        # PCA pela SVD
        _, _, Vt = np.linalg.svd(pts, full_matrices=False)
        v = Vt[0]  # direção principal
        v = v / np.linalg.norm(v)
        if v[2] < 0:  # normaliza sinal
            v = -v

        vz, vy, vx = v[0], v[1], v[2]  # cuidado: v está em (z,y,x)
        # Converte para (x,y,z) para ângulos
        vxyz = np.array([vx, vy, vz], dtype=float)
        vxyz /= np.linalg.norm(vxyz)

        theta = np.degrees(np.arccos(np.clip(vxyz[0], -1.0, 1.0)))  # em relação a X
        phi = np.degrees(np.arctan2(vxyz[1], vxyz[2]))              # azimute no plano YX

        linhas.append({
            "label": int(r.label),
            "vx": float(vxyz[0]),
            "vy": float(vxyz[1]),
            "vz": float(vxyz[2]),
            "theta_deg": float(theta),
            "phi_deg": float(phi),
        })
    return pd.DataFrame(linhas)


# Executa métricas
# df_tamanho = calcular_tamanho_fibras(labeled_fibers, min_voxels=2)
df_orientacao = calcular_orientacao_fibras(labeled_fibers, min_voxels=2)

In [ ]:
df_tamanho = calculate_fiber_length(labeled_skeleton)
df_orientacao = calculate_fiber_orientation(labeled_skeleton)

In [ ]:
# Histogramas
if not df_tamanho.empty:
    plt.figure(dpi=150)
    plt.hist(df_tamanho["n_voxels"], bins=100, color="steelblue", edgecolor="black")
    plt.title("Histogram of the voxel count")
    plt.xlabel("# voxels per fiber")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(dpi=150)
    plt.hist(df_tamanho["length"], bins=100, color="steelblue", edgecolor="black")
    plt.title("Histogram of the fiber length")
    plt.xlabel("length (voxels unit)")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

if not df_orientacao.empty:
    plt.figure(dpi=150)
    plt.hist(df_orientacao["ang_to_x"], bins=np.linspace(0, 180, 90), color="#ff7f0e", edgecolor="black")
    plt.title("Orientação: θ (ângulo com eixo X)")
    plt.xlabel("θ (degrees)")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(dpi=150)
    plt.hist(df_orientacao["ang_to_y"], bins=np.linspace(0, 180, 90), color="#ff7f0e", edgecolor="black")
    plt.title("Orientação: θ (ângulo com eixo Y)")
    plt.xlabel("θ (degrees)")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(dpi=150)
    plt.hist(df_orientacao["ang_to_z"], bins=np.linspace(0, 180, 90), color="#ff7f0e", edgecolor="black")
    plt.title("Orientação: θ (ângulo com eixo Z)")
    plt.xlabel("θ (degrees)")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

# Visualização rápida dos primeiros resultados
print("Amostra df_tamanho:")
print(df_tamanho.head())
print("\nAmostra df_orientacao:")
print(df_orientacao.head())

In [ ]:
def iqr_filter(df, columns, k=1.5, verbose=True):
    """
    Filtra outliers (IQR) em múltiplas colunas.
    Mantém apenas linhas onde cada coluna está dentro de [Q1 - k*IQR, Q3 + k*IQR].
    """
    if df is None or df.empty:
        return df
    mask = np.ones(len(df), dtype=bool)
    stats = {}
    for col in columns:
        if col not in df.columns:
            continue
        series = df[col].dropna()
        if series.empty:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        low = q1 - k * iqr
        high = q3 + k * iqr
        stats[col] = (low, high)
        mask &= (df[col] >= low) & (df[col] <= high)
    if verbose:
        kept = mask.sum()
        print(f"[IQR] Colunas: {columns}  Mantidas: {kept}/{len(df)} linhas")
        for c,(lo,hi) in stats.items():
            print(f"  - {c}: intervalo [{lo:.3f}, {hi:.3f}]")
    return df[mask].reset_index(drop=True)

df_tamanho_filtered = iqr_filter(df_tamanho, ["n_voxels","length"])
df_orientacao_filtered = iqr_filter(df_orientacao, ["ang_to_x","ang_to_y","ang_to_z"])

In [ ]:
df_tamanho_filtered = calculate_fiber_length(remove_small_objects(labeled_skeleton, min_size=10))
df_orientacao_filtered = calculate_fiber_orientation(remove_small_objects(labeled_skeleton, min_size=10))

In [ ]:
# Histogramas
if not df_tamanho_filtered.empty:
    plt.figure(dpi=150)
    plt.hist(df_tamanho_filtered["n_voxels"], bins=100, color="steelblue", edgecolor="black")
    plt.title("Histogram of the voxel count")
    plt.xlabel("# voxels per fiber")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(dpi=150)
    plt.hist(df_tamanho_filtered["length"], bins=100, color="steelblue", edgecolor="black")
    plt.title("Histogram of the fiber length")
    plt.xlabel("length (voxels unit)")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

if not df_orientacao_filtered.empty:
    plt.figure(dpi=150)
    plt.hist(df_orientacao_filtered["ang_to_x"], bins=np.linspace(0, 180, 90), color="#ff7f0e", edgecolor="black")
    plt.title("Orientação: θ (ângulo com eixo X)")
    plt.xlabel("θ (degrees)")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(dpi=150)
    plt.hist(df_orientacao_filtered["ang_to_y"], bins=np.linspace(0, 180, 90), color="#ff7f0e", edgecolor="black")
    plt.title("Orientação: θ (ângulo com eixo Y)")
    plt.xlabel("θ (degrees)")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(dpi=150)
    plt.hist(df_orientacao_filtered["ang_to_z"], bins=np.linspace(0, 180, 90), color="#ff7f0e", edgecolor="black")
    plt.title("Orientação: θ (ângulo com eixo Z)")
    plt.xlabel("θ (degrees)")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.show()

# Visualização rápida dos primeiros resultados
print("Amostra df_tamanho_filtered:")
print(df_tamanho_filtered.head())
print("\nAmostra df_orientacao_filtered:")
print(df_orientacao_filtered.head())